# 01 - Data Understanding

## Objective

The purpose of this notebook is to inspect and understand the raw SMARD
electricity-market datasets before performing any cleaning or transformation.

The analysis covers Germany from 2021 to 2025 at hourly resolution.

### Datasets

1. Actual electricity generation by energy source
2. Actual electricity consumption
3. Wholesale electricity prices

### Questions for the initial inspection

- What is the structure of each dataset?
- How many observations and variables are available?
- What do the columns represent?
- What data types were imported?
- Are there missing or unusual values?
- Is the time period complete?
- Are timestamps consistent across datasets?
- Are there obvious data-quality issues?

No data will be modified during this stage.

In [3]:
import pandas as pd
import numpy as np

## 1. Loading the Raw Data

The original SMARD files use semicolons as separators and commas as decimal
separators. The raw files are loaded without modifying their contents.

In [5]:
generation = pd.read_csv(
    "../data/raw/Realisierte_Erzeugung_2021_2025_Stunde.csv",
    sep=";",
    decimal=","
)

In [1]:
print("hello")

hello


In [7]:
generation.head()

,Datum von,Datum bis,Biomasse [MWh] Berechnete Auflösungen,Wasserkraft [MWh] Berechnete Auflösungen,Wind Offshore [MWh] Berechnete Auflösungen,Wind Onshore [MWh] Berechnete Auflösungen,Photovoltaik [MWh] Berechnete Auflösungen,Sonstige Erneuerbare [MWh] Berechnete Auflösungen,Kernenergie [MWh] Berechnete Auflösungen,Braunkohle [MWh] Berechnete Auflösungen,Steinkohle [MWh] Berechnete Auflösungen,Erdgas [MWh] Berechnete Auflösungen,Pumpspeicher [MWh] Berechnete Auflösungen,Sonstige Konventionelle [MWh] Berechnete Auflösungen
0,01.01.2021 00:00,01.01.2021 01:00,"4.481,00","1.203,00","383,00","3.928,25","1,00",213.00,"8.144,75","11.608,50","3.443,75","6.923,00","347,25","1.637,50"
1,01.01.2021 01:00,01.01.2021 02:00,"4.453,00","1.192,75","394,50","3.528,25","1,00",213.25,"8.150,25","11.602,75","3.044,75","6.688,00","518,50","1.636,00"
2,01.01.2021 02:00,01.01.2021 03:00,"4.440,50","1.161,00","305,25","3.198,00","1,00",216.50,"8.156,50","11.758,50","3.067,25","6.586,00","170,00","1.630,00"
3,01.01.2021 03:00,01.01.2021 04:00,"4.424,75","1.177,75","319,25","2.768,75","1,00",218.00,"8.153,75","12.337,50","2.852,50","6.396,50","2,00","1.621,50"
4,01.01.2021 04:00,01.01.2021 05:00,"4.430,75","1.153,25","296,25","2.462,25","1,00",221.50,"8.150,50","12.395,00","2.713,25","6.333,00","6,00","1.621,50"


In [9]:
generation.shape

(43824, 14)

In [11]:
generation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 14 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Datum von                                             43824 non-null  object 
 1   Datum bis                                             43824 non-null  object 
 2   Biomasse [MWh] Berechnete Auflösungen                 43824 non-null  object 
 3   Wasserkraft [MWh] Berechnete Auflösungen              43824 non-null  object 
 4   Wind Offshore [MWh] Berechnete Auflösungen            43824 non-null  object 
 5   Wind Onshore [MWh] Berechnete Auflösungen             43824 non-null  object 
 6   Photovoltaik [MWh] Berechnete Auflösungen             43824 non-null  object 
 7   Sonstige Erneuerbare [MWh] Berechnete Auflösungen     43824 non-null  float64
 8   Kernenergie [MWh] Berechnete Auflösungen              43

### Initial observations

- The generation dataset contains **43,824 hourly observations** and **14 columns**.
- Two columns represent the start and end timestamps of each observation (`Datum von` and `Datum bis`).
- The remaining 12 columns represent electricity generation from different energy sources.
- All columns contain 43,824 non-null observations according to the initial import.
- Most generation columns were imported as `object` rather than numeric values.
- Only `Sonstige Erneuerbare` was automatically recognized as `float64`.
- This suggests a number-formatting issue in the raw SMARD data. German-formatted values such as `4.481,00` use a period as the thousands separator and a comma as the decimal separator.
- The timestamp columns were also imported as `object` and will later need to be converted to datetime values.

- The dataset contains exactly **43,824 hourly observations**, matching the expected number of hours from 2021–2025 (including the leap year 2024). This indicates that the dataset is complete at the overall row-count level.

In [15]:
consumption = pd.read_csv(
    "../data/raw/Realisierter_Stromverbrauch_2021_2025_Stunde.csv",
    sep=";",
    decimal=","
)

In [17]:
consumption.head

<bound method NDFrame.head of               Datum von         Datum bis  \
0      01.01.2021 00:00  01.01.2021 01:00   
1      01.01.2021 01:00  01.01.2021 02:00   
2      01.01.2021 02:00  01.01.2021 03:00   
3      01.01.2021 03:00  01.01.2021 04:00   
4      01.01.2021 04:00  01.01.2021 05:00   
...                 ...               ...   
43819  31.12.2025 19:00  31.12.2025 20:00   
43820  31.12.2025 20:00  31.12.2025 21:00   
43821  31.12.2025 21:00  31.12.2025 22:00   
43822  31.12.2025 22:00  31.12.2025 23:00   
43823  31.12.2025 23:00  01.01.2026 00:00   

      Netzlast [MWh] Berechnete Auflösungen  \
0                                 44.569,25   
1                                 42.806,00   
2                                 41.049,75   
3                                 40.233,75   
4                                 40.210,50   
...                                     ...   
43819                             55.505,61   
43820                             52.297,32   
43821 

In [19]:
consumption.shape

(43824, 6)

In [21]:
consumption.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 6 columns):
 #   Column                                                    Non-Null Count  Dtype 
---  ------                                                    --------------  ----- 
 0   Datum von                                                 43824 non-null  object
 1   Datum bis                                                 43824 non-null  object
 2   Netzlast [MWh] Berechnete Auflösungen                     43824 non-null  object
 3   Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen  43824 non-null  object
 4   Pumpspeicher [MWh] Berechnete Auflösungen                 43824 non-null  object
 5   Residuallast [MWh] Berechnete Auflösungen                 43824 non-null  object
dtypes: object(6)
memory usage: 2.0+ MB


### Initial observations - Consumption Data

- Shape: **43,824 rows × 6 columns**
- The row count matches the expected number of hourly observations from 2021–2025.
- No missing values are detected.
- All columns are currently stored as `object`.
- The two date columns will need to be converted to datetime.
- Numerical electricity-consumption columns will need datatype conversion during cleaning.

In [25]:
prices = pd.read_csv(
    "../data/raw/Gro_handelspreise_2021_2025_Stunde.csv",
    sep=";",
    decimal=","
)

C:\Users\megha\AppData\Local\Temp\ipykernel_49728\3860847318.py:1: DtypeWarning: Columns (7,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  prices = pd.read_csv(


In [27]:
prices.head()

,Datum von,Datum bis,Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen,∅ Anrainer DE/LU [€/MWh] Berechnete Auflösungen,Belgien [€/MWh] Berechnete Auflösungen,Dänemark 1 [€/MWh] Berechnete Auflösungen,Dänemark 2 [€/MWh] Berechnete Auflösungen,Frankreich [€/MWh] Berechnete Auflösungen,Niederlande [€/MWh] Berechnete Auflösungen,Norwegen 2 [€/MWh] Berechnete Auflösungen,Österreich [€/MWh] Berechnete Auflösungen,Polen [€/MWh] Berechnete Auflösungen,Schweden 4 [€/MWh] Berechnete Auflösungen,Schweiz [€/MWh] Berechnete Auflösungen,Tschechien [€/MWh] Berechnete Auflösungen,DE/AT/LU [€/MWh] Berechnete Auflösungen,Italien (Nord) [€/MWh] Berechnete Auflösungen,Slowenien [€/MWh] Berechnete Auflösungen,Ungarn [€/MWh] Berechnete Auflösungen
0,01.01.2021 00:00,01.01.2021 01:00,50.87,44.17,50.87,50.87,50.87,"50,87",50.87,24.95,50.87,34.18,24.95,50.98,45.54,-,50.87,"50,87","45,54"
1,01.01.2021 01:00,01.01.2021 02:00,48.19,41.63,48.19,48.19,48.19,"48,19",48.19,24.35,48.19,32.61,24.35,45.94,41.59,-,48.19,"48,19","41,59"
2,01.01.2021 02:00,01.01.2021 03:00,44.68,39.38,44.68,44.68,44.68,"44,68",44.68,23.98,44.68,32.58,23.98,44.53,40.05,-,44.68,"44,68","40,05"
3,01.01.2021 03:00,01.01.2021 04:00,42.92,37.73,42.92,42.92,42.92,"42,92",42.92,23.72,42.92,32.20,23.72,41.01,36.90,-,42.92,"42,92","36,90"
4,01.01.2021 04:00,01.01.2021 05:00,40.39,35.70,40.39,40.39,40.39,"40,39",40.39,23.73,40.39,29.16,23.73,39.23,34.47,-,40.39,"40,39","34,47"


In [29]:
prices.shape

(43824, 19)

In [31]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 19 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Datum von                                             43824 non-null  object 
 1   Datum bis                                             43824 non-null  object 
 2   Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen  43824 non-null  float64
 3   ∅ Anrainer DE/LU [€/MWh] Berechnete Auflösungen       43824 non-null  float64
 4   Belgien [€/MWh] Berechnete Auflösungen                43824 non-null  float64
 5   Dänemark 1 [€/MWh] Berechnete Auflösungen             43824 non-null  float64
 6   Dänemark 2 [€/MWh] Berechnete Auflösungen             43824 non-null  float64
 7   Frankreich [€/MWh] Berechnete Auflösungen             43824 non-null  object 
 8   Niederlande [€/MWh] Berechnete Auflösungen            43

In [33]:
prices.select_dtypes(include="object").columns

Index(['Datum von', 'Datum bis', 'Frankreich [€/MWh] Berechnete Auflösungen',
       'DE/AT/LU [€/MWh] Berechnete Auflösungen',
       'Slowenien [€/MWh] Berechnete Auflösungen',
       'Ungarn [€/MWh] Berechnete Auflösungen'],
      dtype='object')

In [35]:
prices["Frankreich [€/MWh] Berechnete Auflösungen"].unique()[:20]

array(['50,87', '48,19', '44,68', '42,92', '40,39', '40,20', '39,63',
       '40,09', '41,27', '44,88', '45,00', '47,20', '50,78', '45,49',
       '44,73', '46,59', '52,99', '60,26', '60,61', '60,36'], dtype=object)

In [37]:
france_numeric = pd.to_numeric(
    prices["Frankreich [€/MWh] Berechnete Auflösungen"].str.replace(",", "."),
    errors="coerce"
)

france_numeric.isna().sum()

11059

In [39]:
prices.loc[
    france_numeric.isna(),
    "Frankreich [€/MWh] Berechnete Auflösungen"
].value_counts(dropna=False)

Frankreich [€/MWh] Berechnete Auflösungen
 0.00      251
-0.01      196
-0.02       52
 35.01      25
 35.00      23
          ... 
 133.98      1
 127.42      1
 88.29       1
 85.53       1
 80.35       1
Name: count, Length: 7246, dtype: int64

### Price data quality observation

Several foreign bidding-zone price columns are stored as `object` and contain inconsistent numeric formatting. Since the core analysis focuses on the German/Luxembourg electricity market, these columns are not required for the primary analysis and will not be cleaned at this stage.

The Germany/Luxembourg wholesale price series contains 43,824 non-null observations and is already stored as a numeric variable (`float64`).

## 2. Variable Selection

After inspecting the three SMARD datasets, the variables required for the analysis are selected. The original raw datasets remain unchanged; variable selection is performed only for the analytical workflow.

### Generation Data

All available electricity generation technologies will be retained because they are relevant for analysing Germany's generation mix and the relationship between renewable and conventional generation and electricity prices.

The dataset therefore retains:

- Biomass
- Hydropower
- Offshore wind
- Onshore wind
- Photovoltaics
- Other renewables
- Nuclear
- Lignite
- Hard coal
- Natural gas
- Pumped storage
- Other conventional generation

`Datum von` will be used as the hourly timestamp. `Datum bis` will be removed only after verifying that each observation represents a one-hour interval.

### Consumption Data

All available consumption-related variables will initially be retained:

- Grid load (`Netzlast`)
- Grid load including pumped storage
- Pumped-storage consumption
- Residual load

Keeping these variables provides flexibility for later analyses of electricity demand, renewable integration, and residual load.

As with the generation data, `Datum von` will serve as the timestamp and `Datum bis` will only be removed after validating the hourly intervals.

### Wholesale Price Data

Germany/Luxembourg is the primary electricity market analysed in this project. Selected neighbouring markets will also be retained to allow comparison of wholesale electricity prices across interconnected European markets.

The selected markets are:

- Germany/Luxembourg
- France
- Netherlands
- Belgium
- Denmark (DK1)
- Austria
- Poland
- Switzerland

Other bidding zones are not required for the current analytical scope and will remain available in the original raw dataset.

### Price Data Quality Observation

During the initial inspection, several foreign bidding-zone price columns were stored as `object` rather than numeric variables. This indicates inconsistent numeric representation within some of these columns.

The Germany/Luxembourg wholesale price series is already stored as `float64`. Selected neighbouring-market columns that are not numeric will be investigated and cleaned during the data-preparation stage before they are used for comparison.

## 3. Data Cleaning and Validation

Before the datasets are merged and analysed, their structure and data quality need to be validated. This includes checking timestamps, numeric formats, missing values, duplicate observations, and the completeness of the hourly time series.

### 3.1 Timestamp Validation

The columns `Datum von` and `Datum bis` represent the beginning and end of each observation period. They are currently stored as text (`object`), which prevents proper time-series operations.

Both columns will therefore be converted to Python `datetime` values.

Before removing `Datum bis`, the duration between `Datum von` and `Datum bis` will be checked. If each observation consistently represents a one-hour interval, `Datum bis` does not provide additional information for the analysis and can later be removed.

In [45]:
for df in [generation, consumption, prices]:
    df["Datum von"] = pd.to_datetime(
        df["Datum von"],
        format="%d.%m.%Y %H:%M"
    )
    
    df["Datum bis"] = pd.to_datetime(
        df["Datum bis"],
        format="%d.%m.%Y %H:%M"
    )

In [47]:
generation[["Datum von", "Datum bis"]].dtypes

Datum von    datetime64[ns]
Datum bis    datetime64[ns]
dtype: object

#### Checking the Observation Interval

Before removing `Datum bis`, the duration of each observation is calculated as the difference between the end and start timestamps.

If all observations have a duration of one hour, `Datum von` alone is sufficient to identify each hourly observation.

In [50]:
for name, df in {
    "Generation": generation,
    "Consumption": consumption,
    "Prices": prices
}.items():
    
    interval = df["Datum bis"] - df["Datum von"]
    
    print(name)
    print(interval.value_counts())
    print()

Generation
0 days 01:00:00    43824
Name: count, dtype: int64

Consumption
0 days 01:00:00    43824
Name: count, dtype: int64

Prices
0 days 01:00:00    43824
Name: count, dtype: int64



#### Result

All 43,824 observations in each of the three datasets represent exactly one-hour intervals.

Therefore, `Datum bis` is redundant for the planned hourly analysis. `Datum von` will be retained as the timestamp identifying each observation, while `Datum bis` can be removed during dataset preparation.

### 3.2 Timestamp Coverage and Uniqueness

After validating the duration of each observation, the timestamp series is checked for completeness and uniqueness.

The following checks are performed:

- Identify the first and last timestamp in each dataset.
- Check whether duplicate timestamps exist.
- Compare the timestamp coverage across generation, consumption, and price data.
- Investigate irregularities in the hourly sequence, particularly those related to daylight-saving time changes.

These checks ensure that the datasets can later be merged reliably using the hourly timestamp.

In [54]:
for name, df in {
    "Generation": generation,
    "Consumption": consumption,
    "Prices": prices
}.items():
    
    print(name)
    print("Start:", df["Datum von"].min())
    print("End:  ", df["Datum von"].max())
    print("Rows: ", len(df))
    print()

Generation
Start: 2021-01-01 00:00:00
End:   2025-12-31 23:00:00
Rows:  43824

Consumption
Start: 2021-01-01 00:00:00
End:   2025-12-31 23:00:00
Rows:  43824

Prices
Start: 2021-01-01 00:00:00
End:   2025-12-31 23:00:00
Rows:  43824



In [56]:
for name, df in {
    "Generation": generation,
    "Consumption": consumption,
    "Prices": prices
}.items():

    duplicates = df["Datum von"].duplicated().sum()

    print(name, "- Duplicate timestamps:", duplicates)

Generation - Duplicate timestamps: 5
Consumption - Duplicate timestamps: 5
Prices - Duplicate timestamps: 5


In [58]:
generation.loc[
    generation["Datum von"].duplicated(keep=False),
    ["Datum von", "Datum bis"]
]

,Datum von,Datum bis
7273,2021-10-31 02:00:00,2021-10-31 03:00:00
7274,2021-10-31 02:00:00,2021-10-31 03:00:00
16009,2022-10-30 02:00:00,2022-10-30 03:00:00
16010,2022-10-30 02:00:00,2022-10-30 03:00:00
24745,2023-10-29 02:00:00,2023-10-29 03:00:00
24746,2023-10-29 02:00:00,2023-10-29 03:00:00
33481,2024-10-27 02:00:00,2024-10-27 03:00:00
33482,2024-10-27 02:00:00,2024-10-27 03:00:00
42217,2025-10-26 02:00:00,2025-10-26 03:00:00
42218,2025-10-26 02:00:00,2025-10-26 03:00:00


#### Duplicate Timestamp Observation

Five duplicated `Datum von` timestamps were identified in each dataset, corresponding to one duplicated hour per year from 2021 to 2025.

Inspection showed that all duplicated timestamps occur at 02:00 on the annual transition from Central European Summer Time (CEST) to Central European Time (CET), when the clock is moved back by one hour.

These observations therefore represent valid daylight-saving-time repetitions rather than erroneous duplicate records and should not be removed.

This also means that the local timestamp `Datum von` alone is not a unique identifier for every hourly observation during the daylight-saving-time transition.

#### Checking Gaps in the Hourly Time Series

In addition to repeated hours during the autumn daylight-saving-time transition, the switch from standard time (CET) to summer time (CEST) in spring causes the clock to move forward by one hour.

Therefore, an apparent two-hour difference between consecutive local timestamps may be expected once per year.

The differences between consecutive timestamps are examined to distinguish expected daylight-saving-time transitions from potential missing observations.

In [65]:
for name, df in {
    "Generation": generation,
    "Consumption": consumption,
    "Prices": prices
}.items():

    time_differences = df["Datum von"].diff()

    print(name)
    print(time_differences.value_counts().sort_index())
    print()

Generation
Datum von
0 days 00:00:00        5
0 days 01:00:00    43813
0 days 02:00:00        5
Name: count, dtype: int64

Consumption
Datum von
0 days 00:00:00        5
0 days 01:00:00    43813
0 days 02:00:00        5
Name: count, dtype: int64

Prices
Datum von
0 days 00:00:00        5
0 days 01:00:00    43813
0 days 02:00:00        5
Name: count, dtype: int64



#### Timestamp Validation Summary

Generation, consumption, and wholesale price data show the same timestamp structure from 1 January 2021 to 31 December 2025.

All three datasets contain:

- 43,824 hourly observations.
- 5 repeated local timestamps caused by the autumn daylight-saving-time transition.
- 5 two-hour timestamp differences caused by the spring daylight-saving-time transition.
- No additional unexpected gaps in the timestamp sequence.

The repeated observations are valid market observations and will not be removed. Special care will be taken when merging the datasets because the local timestamp alone is not unique during the autumn daylight-saving-time transition.

In [68]:
col = "Biomasse [MWh] Berechnete Auflösungen"

for value in generation[col].head(10):
    print(repr(value), type(value))

'4.481,00' <class 'str'>
'4.453,00' <class 'str'>
'4.440,50' <class 'str'>
'4.424,75' <class 'str'>
'4.430,75' <class 'str'>
'4.451,00' <class 'str'>
'4.491,25' <class 'str'>
'4.561,25' <class 'str'>
'4.646,25' <class 'str'>
'4.655,25' <class 'str'>


In [70]:
for col in generation.columns[2:]:
    print(
        col,
        "→",
        generation[col].dtype,
        "| Example:",
        repr(generation[col].iloc[0]),
        "| Python type:",
        type(generation[col].iloc[0]).__name__
    )

Biomasse [MWh] Berechnete Auflösungen → object | Example: '4.481,00' | Python type: str
Wasserkraft [MWh] Berechnete Auflösungen → object | Example: '1.203,00' | Python type: str
Wind Offshore [MWh] Berechnete Auflösungen → object | Example: '383,00' | Python type: str
Wind Onshore [MWh] Berechnete Auflösungen → object | Example: '3.928,25' | Python type: str
Photovoltaik [MWh] Berechnete Auflösungen → object | Example: '1,00' | Python type: str
Sonstige Erneuerbare [MWh] Berechnete Auflösungen → float64 | Example: 213.0 | Python type: float64
Kernenergie [MWh] Berechnete Auflösungen → object | Example: '8.144,75' | Python type: str
Braunkohle [MWh] Berechnete Auflösungen → object | Example: '11.608,50' | Python type: str
Steinkohle [MWh] Berechnete Auflösungen → object | Example: '3.443,75' | Python type: str
Erdgas [MWh] Berechnete Auflösungen → object | Example: '6.923,00' | Python type: str
Pumpspeicher [MWh] Berechnete Auflösungen → object | Example: '347,25' | Python type: str
So

#### Generation Data Type Observation

Inspection of the generation variables shows that most MWh values are stored as strings using German number formatting.

For example, `4.481,00` represents `4481.00` MWh, where the period is used as the thousands separator and the comma as the decimal separator.

Eleven generation variables are stored as `object`, while `Sonstige Erneuerbare` is already stored as `float64`. Therefore, only the string-based generation columns need to be converted to numeric values.

In [73]:
generation_object_cols = generation.columns[
    generation.dtypes == "object"
]

generation_object_cols

Index(['Biomasse [MWh] Berechnete Auflösungen',
       'Wasserkraft [MWh] Berechnete Auflösungen',
       'Wind Offshore [MWh] Berechnete Auflösungen',
       'Wind Onshore [MWh] Berechnete Auflösungen',
       'Photovoltaik [MWh] Berechnete Auflösungen',
       'Kernenergie [MWh] Berechnete Auflösungen',
       'Braunkohle [MWh] Berechnete Auflösungen',
       'Steinkohle [MWh] Berechnete Auflösungen',
       'Erdgas [MWh] Berechnete Auflösungen',
       'Pumpspeicher [MWh] Berechnete Auflösungen',
       'Sonstige Konventionelle [MWh] Berechnete Auflösungen'],
      dtype='object')

In [75]:
for col in generation_object_cols:
    generation[col] = pd.to_numeric(
        generation[col]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

In [77]:
generation.dtypes

Datum von                                               datetime64[ns]
Datum bis                                               datetime64[ns]
Biomasse [MWh] Berechnete Auflösungen                          float64
Wasserkraft [MWh] Berechnete Auflösungen                       float64
Wind Offshore [MWh] Berechnete Auflösungen                     float64
Wind Onshore [MWh] Berechnete Auflösungen                      float64
Photovoltaik [MWh] Berechnete Auflösungen                      float64
Sonstige Erneuerbare [MWh] Berechnete Auflösungen              float64
Kernenergie [MWh] Berechnete Auflösungen                       float64
Braunkohle [MWh] Berechnete Auflösungen                        float64
Steinkohle [MWh] Berechnete Auflösungen                        float64
Erdgas [MWh] Berechnete Auflösungen                            float64
Pumpspeicher [MWh] Berechnete Auflösungen                      float64
Sonstige Konventionelle [MWh] Berechnete Auflösungen           float64
dtype:

In [79]:
generation.head()

,Datum von,Datum bis,Biomasse [MWh] Berechnete Auflösungen,Wasserkraft [MWh] Berechnete Auflösungen,Wind Offshore [MWh] Berechnete Auflösungen,Wind Onshore [MWh] Berechnete Auflösungen,Photovoltaik [MWh] Berechnete Auflösungen,Sonstige Erneuerbare [MWh] Berechnete Auflösungen,Kernenergie [MWh] Berechnete Auflösungen,Braunkohle [MWh] Berechnete Auflösungen,Steinkohle [MWh] Berechnete Auflösungen,Erdgas [MWh] Berechnete Auflösungen,Pumpspeicher [MWh] Berechnete Auflösungen,Sonstige Konventionelle [MWh] Berechnete Auflösungen
0,2021-01-01 00:00:00,2021-01-01 01:00:00,4481.00,1203.00,383.00,3928.25,1.0,213.00,8144.75,11608.50,3443.75,6923.0,347.25,1637.5
1,2021-01-01 01:00:00,2021-01-01 02:00:00,4453.00,1192.75,394.50,3528.25,1.0,213.25,8150.25,11602.75,3044.75,6688.0,518.50,1636.0
2,2021-01-01 02:00:00,2021-01-01 03:00:00,4440.50,1161.00,305.25,3198.00,1.0,216.50,8156.50,11758.50,3067.25,6586.0,170.00,1630.0
3,2021-01-01 03:00:00,2021-01-01 04:00:00,4424.75,1177.75,319.25,2768.75,1.0,218.00,8153.75,12337.50,2852.50,6396.5,2.00,1621.5
4,2021-01-01 04:00:00,2021-01-01 05:00:00,4430.75,1153.25,296.25,2462.25,1.0,221.50,8150.50,12395.00,2713.25,6333.0,6.00,1621.5


#### Generation Conversion Validation

After converting the string-based generation variables to numeric values, the results are validated to ensure that the conversion was successful.

The data types are checked to confirm that all generation variables are numeric. In addition, missing values are counted because values that could not be converted would have been replaced with `NaN`.

In [82]:
generation.isna().sum()

Datum von                                                   0
Datum bis                                                   0
Biomasse [MWh] Berechnete Auflösungen                       0
Wasserkraft [MWh] Berechnete Auflösungen                    0
Wind Offshore [MWh] Berechnete Auflösungen                  0
Wind Onshore [MWh] Berechnete Auflösungen                   0
Photovoltaik [MWh] Berechnete Auflösungen                   0
Sonstige Erneuerbare [MWh] Berechnete Auflösungen           0
Kernenergie [MWh] Berechnete Auflösungen                16836
Braunkohle [MWh] Berechnete Auflösungen                     0
Steinkohle [MWh] Berechnete Auflösungen                     0
Erdgas [MWh] Berechnete Auflösungen                         0
Pumpspeicher [MWh] Berechnete Auflösungen                   0
Sonstige Konventionelle [MWh] Berechnete Auflösungen        0
dtype: int64

In [84]:
nuclear_col = "Kernenergie [MWh] Berechnete Auflösungen"

generation.loc[
    generation[nuclear_col].isna(),
    ["Datum von", nuclear_col]
].head(10)

,Datum von,Kernenergie [MWh] Berechnete Auflösungen
26988,2024-01-30 12:00:00,NaN
26989,2024-01-30 13:00:00,NaN
26990,2024-01-30 14:00:00,NaN
26991,2024-01-30 15:00:00,NaN
26992,2024-01-30 16:00:00,NaN
26993,2024-01-30 17:00:00,NaN
26994,2024-01-30 18:00:00,NaN
26995,2024-01-30 19:00:00,NaN
26996,2024-01-30 20:00:00,NaN
26997,2024-01-30 21:00:00,NaN


In [86]:
generation.loc[
    generation[nuclear_col].isna(),
    "Datum von"
].min()

Timestamp('2024-01-30 12:00:00')

In [88]:
raw_generation_check = pd.read_csv(
    "../data/raw/Realisierte_Erzeugung_2021_2025_Stunde.csv",
    sep=";",
    decimal=","
)

nuclear_col = "Kernenergie [MWh] Berechnete Auflösungen"

raw_generation_check.loc[
    26980:27000,
    ["Datum von", nuclear_col]
]

,Datum von,Kernenergie [MWh] Berechnete Auflösungen
26980,30.01.2024 04:00,"0,00"
26981,30.01.2024 05:00,"0,00"
26982,30.01.2024 06:00,"0,00"
26983,30.01.2024 07:00,"0,00"
26984,30.01.2024 08:00,"0,00"
26985,30.01.2024 09:00,"0,00"
26986,30.01.2024 10:00,"0,00"
26987,30.01.2024 11:00,"0,00"
26988,30.01.2024 12:00,-
26989,30.01.2024 13:00,-


#### Nuclear Generation Missing Values

The numeric conversion produced 16,836 missing values in the nuclear generation variable. Inspection of the original SMARD data showed that these observations are represented by `-` in the source file rather than by numerical values.

The source reports nuclear generation as `0,00` after the end of German nuclear electricity production before later switching to `-` from 30 January 2024 at 12:00.

For the purpose of analysing electricity generation, these post-phase-out observations represent the absence of nuclear generation and are therefore treated as 0 MWh rather than as unknown missing observations.

In [91]:
generation[nuclear_col] = generation[nuclear_col].fillna(0)

In [93]:
generation[nuclear_col].isna().sum()

0

In [95]:
generation.isna().sum()

Datum von                                               0
Datum bis                                               0
Biomasse [MWh] Berechnete Auflösungen                   0
Wasserkraft [MWh] Berechnete Auflösungen                0
Wind Offshore [MWh] Berechnete Auflösungen              0
Wind Onshore [MWh] Berechnete Auflösungen               0
Photovoltaik [MWh] Berechnete Auflösungen               0
Sonstige Erneuerbare [MWh] Berechnete Auflösungen       0
Kernenergie [MWh] Berechnete Auflösungen                0
Braunkohle [MWh] Berechnete Auflösungen                 0
Steinkohle [MWh] Berechnete Auflösungen                 0
Erdgas [MWh] Berechnete Auflösungen                     0
Pumpspeicher [MWh] Berechnete Auflösungen               0
Sonstige Konventionelle [MWh] Berechnete Auflösungen    0
dtype: int64

### 3.4 Consumption Data Validation and Cleaning

The consumption dataset contains four electricity-demand-related variables. During the initial inspection, these variables were stored as `object` rather than numeric values.

As with the generation data, the formatting of these variables is first inspected before conversion. String-based numerical values will then be converted to numeric data types, followed by a missing-value check to verify that the conversion was successful.

In [98]:
for col in consumption.columns[2:]:
    print(
        col,
        "→",
        consumption[col].dtype,
        "| Example:",
        repr(consumption[col].iloc[0]),
        "| Python type:",
        type(consumption[col].iloc[0]).__name__
    )

Netzlast [MWh] Berechnete Auflösungen → object | Example: '44.569,25' | Python type: str
Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen → object | Example: '44.957,50' | Python type: str
Pumpspeicher [MWh] Berechnete Auflösungen → object | Example: '388,25' | Python type: str
Residuallast [MWh] Berechnete Auflösungen → object | Example: '40.257,00' | Python type: str


In [100]:
consumption_object_cols = consumption.columns[
    consumption.dtypes == "object"
]

consumption_object_cols

Index(['Netzlast [MWh] Berechnete Auflösungen',
       'Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen',
       'Pumpspeicher [MWh] Berechnete Auflösungen',
       'Residuallast [MWh] Berechnete Auflösungen'],
      dtype='object')

In [102]:
for col in consumption_object_cols:
    consumption[col] = pd.to_numeric(
        consumption[col]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

In [104]:
consumption.dtypes

Datum von                                                   datetime64[ns]
Datum bis                                                   datetime64[ns]
Netzlast [MWh] Berechnete Auflösungen                              float64
Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen           float64
Pumpspeicher [MWh] Berechnete Auflösungen                          float64
Residuallast [MWh] Berechnete Auflösungen                          float64
dtype: object

In [106]:
consumption.isna().sum()

Datum von                                                   0
Datum bis                                                   0
Netzlast [MWh] Berechnete Auflösungen                       0
Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen    0
Pumpspeicher [MWh] Berechnete Auflösungen                   0
Residuallast [MWh] Berechnete Auflösungen                   0
dtype: int64

#### Consumption Conversion Result

All four consumption variables were successfully converted from German-formatted strings to numeric values.

No missing values were introduced during the conversion, confirming that all original consumption observations could be converted successfully.

### 3.5 Wholesale Price Data Validation and Cleaning

The wholesale price dataset contains electricity prices for Germany/Luxembourg and several European bidding zones.

The core analysis focuses on Germany/Luxembourg, while selected neighbouring markets are retained for comparative analysis:

- Germany/Luxembourg
- France
- Netherlands
- Belgium
- Denmark (DK1)
- Austria
- Poland
- Switzerland

Because the initial inspection showed that some price columns have different data types and numeric representations, the selected variables are inspected individually before conversion.

In [110]:
for i, col in enumerate(prices.columns):
    print(i, col)

0 Datum von
1 Datum bis
2 Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen
3 ∅ Anrainer DE/LU [€/MWh] Berechnete Auflösungen
4 Belgien [€/MWh] Berechnete Auflösungen
5 Dänemark 1 [€/MWh] Berechnete Auflösungen
6 Dänemark 2 [€/MWh] Berechnete Auflösungen
7 Frankreich [€/MWh] Berechnete Auflösungen
8 Niederlande [€/MWh] Berechnete Auflösungen
9 Norwegen 2 [€/MWh] Berechnete Auflösungen
10 Österreich [€/MWh] Berechnete Auflösungen
11 Polen [€/MWh] Berechnete Auflösungen
12 Schweden 4 [€/MWh] Berechnete Auflösungen
13 Schweiz [€/MWh] Berechnete Auflösungen
14 Tschechien [€/MWh] Berechnete Auflösungen
15 DE/AT/LU [€/MWh] Berechnete Auflösungen
16 Italien (Nord) [€/MWh] Berechnete Auflösungen
17 Slowenien [€/MWh] Berechnete Auflösungen
18 Ungarn [€/MWh] Berechnete Auflösungen


In [112]:
selected_price_cols = [
    prices.columns[2],   # Germany/Luxembourg
    prices.columns[4],   # Belgium
    prices.columns[5],   # Denmark DK1
    prices.columns[7],   # France
    prices.columns[8],   # Netherlands
    prices.columns[10],  # Austria
    prices.columns[11],  # Poland
    prices.columns[13]   # Switzerland
]

for col in selected_price_cols:
    print(
        col,
        "→",
        prices[col].dtype,
        "| Example:",
        repr(prices[col].iloc[0]),
        "| Python type:",
        type(prices[col].iloc[0]).__name__
    )

Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen → float64 | Example: 50.87 | Python type: float64
Belgien [€/MWh] Berechnete Auflösungen → float64 | Example: 50.87 | Python type: float64
Dänemark 1 [€/MWh] Berechnete Auflösungen → float64 | Example: 50.87 | Python type: float64
Frankreich [€/MWh] Berechnete Auflösungen → object | Example: '50,87' | Python type: str
Niederlande [€/MWh] Berechnete Auflösungen → float64 | Example: 50.87 | Python type: float64
Österreich [€/MWh] Berechnete Auflösungen → float64 | Example: 50.87 | Python type: float64
Polen [€/MWh] Berechnete Auflösungen → float64 | Example: 34.18 | Python type: float64
Schweiz [€/MWh] Berechnete Auflösungen → float64 | Example: 50.98 | Python type: float64


In [114]:
france_col = "Frankreich [€/MWh] Berechnete Auflösungen"

prices[france_col].map(type).value_counts()

Frankreich [€/MWh] Berechnete Auflösungen
<class 'str'>      32768
<class 'float'>    11056
Name: count, dtype: int64

In [116]:
def clean_price_value(value):
    if isinstance(value, str):
        value = value.replace(".", "").replace(",", ".")
        return pd.to_numeric(value, errors="coerce")
    
    return value

In [118]:
prices[france_col] = prices[france_col].apply(clean_price_value)

In [120]:
print(prices[france_col].dtype)
print(prices[france_col].isna().sum())
print(prices[france_col].map(type).value_counts())

float64
0
Frankreich [€/MWh] Berechnete Auflösungen
<class 'float'>    43824
Name: count, dtype: int64


#### Wholesale Price Conversion Result

Among the selected electricity markets, Germany/Luxembourg, Belgium, Denmark (DK1), the Netherlands, Austria, Poland, and Switzerland were already stored as numeric variables.

The French price series contained a mixture of string and numeric values. String values using German decimal formatting were converted individually while existing numeric values were preserved.

After conversion, all 43,824 French price observations are stored as numeric values (`float64`) and no missing values were introduced.

In [123]:
prices[selected_price_cols].isna().sum()

Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen    0
Belgien [€/MWh] Berechnete Auflösungen                  0
Dänemark 1 [€/MWh] Berechnete Auflösungen               0
Frankreich [€/MWh] Berechnete Auflösungen               0
Niederlande [€/MWh] Berechnete Auflösungen              0
Österreich [€/MWh] Berechnete Auflösungen               0
Polen [€/MWh] Berechnete Auflösungen                    0
Schweiz [€/MWh] Berechnete Auflösungen                  0
dtype: int64

#### Wholesale Price Validation Summary

All eight selected wholesale electricity price series contain complete observations with no missing values after numeric conversion.

The selected price variables are therefore ready for the preparation of the analytical dataset.

## 4. Preparing the Analytical Datasets

After completing the initial data-quality checks, separate analytical datasets are created from the cleaned source data.

Only variables required for the planned analysis are retained, and the original SMARD column names are replaced with concise English names. This improves readability and provides a consistent structure for the subsequent Python, SQL, and Power BI analyses.

`Datum bis` is no longer required because the previous validation confirmed that every observation represents a one-hour interval. `Datum von` is retained as the primary timestamp.

The repeated local timestamps caused by the autumn daylight-saving-time transition are preserved and will be handled explicitly before the datasets are merged.

In [127]:
generation_clean = generation.drop(columns=["Datum bis"]).copy()

generation_clean.columns

Index(['Datum von', 'Biomasse [MWh] Berechnete Auflösungen',
       'Wasserkraft [MWh] Berechnete Auflösungen',
       'Wind Offshore [MWh] Berechnete Auflösungen',
       'Wind Onshore [MWh] Berechnete Auflösungen',
       'Photovoltaik [MWh] Berechnete Auflösungen',
       'Sonstige Erneuerbare [MWh] Berechnete Auflösungen',
       'Kernenergie [MWh] Berechnete Auflösungen',
       'Braunkohle [MWh] Berechnete Auflösungen',
       'Steinkohle [MWh] Berechnete Auflösungen',
       'Erdgas [MWh] Berechnete Auflösungen',
       'Pumpspeicher [MWh] Berechnete Auflösungen',
       'Sonstige Konventionelle [MWh] Berechnete Auflösungen'],
      dtype='object')

In [129]:
generation_clean = generation_clean.rename(columns={
    "Datum von": "timestamp",
    "Biomasse [MWh] Berechnete Auflösungen": "biomass_mwh",
    "Wasserkraft [MWh] Berechnete Auflösungen": "hydro_mwh",
    "Wind Offshore [MWh] Berechnete Auflösungen": "wind_offshore_mwh",
    "Wind Onshore [MWh] Berechnete Auflösungen": "wind_onshore_mwh",
    "Photovoltaik [MWh] Berechnete Auflösungen": "solar_mwh",
    "Sonstige Erneuerbare [MWh] Berechnete Auflösungen": "other_renewables_mwh",
    "Kernenergie [MWh] Berechnete Auflösungen": "nuclear_mwh",
    "Braunkohle [MWh] Berechnete Auflösungen": "lignite_mwh",
    "Steinkohle [MWh] Berechnete Auflösungen": "hard_coal_mwh",
    "Erdgas [MWh] Berechnete Auflösungen": "gas_mwh",
    "Pumpspeicher [MWh] Berechnete Auflösungen": "pumped_storage_generation_mwh",
    "Sonstige Konventionelle [MWh] Berechnete Auflösungen": "other_conventional_mwh"
})

In [131]:
generation_clean.head()

,timestamp,biomass_mwh,hydro_mwh,wind_offshore_mwh,wind_onshore_mwh,solar_mwh,other_renewables_mwh,nuclear_mwh,lignite_mwh,hard_coal_mwh,gas_mwh,pumped_storage_generation_mwh,other_conventional_mwh
0,2021-01-01 00:00:00,4481.00,1203.00,383.00,3928.25,1.0,213.00,8144.75,11608.50,3443.75,6923.0,347.25,1637.5
1,2021-01-01 01:00:00,4453.00,1192.75,394.50,3528.25,1.0,213.25,8150.25,11602.75,3044.75,6688.0,518.50,1636.0
2,2021-01-01 02:00:00,4440.50,1161.00,305.25,3198.00,1.0,216.50,8156.50,11758.50,3067.25,6586.0,170.00,1630.0
3,2021-01-01 03:00:00,4424.75,1177.75,319.25,2768.75,1.0,218.00,8153.75,12337.50,2852.50,6396.5,2.00,1621.5
4,2021-01-01 04:00:00,4430.75,1153.25,296.25,2462.25,1.0,221.50,8150.50,12395.00,2713.25,6333.0,6.00,1621.5


In [133]:
consumption_clean = consumption.drop(columns=["Datum bis"]).copy()

In [135]:
consumption_clean = consumption_clean.rename(columns={
    "Datum von": "timestamp",
    "Netzlast [MWh] Berechnete Auflösungen": "load_mwh",
    "Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen": "load_incl_pumped_storage_mwh",
    "Pumpspeicher [MWh] Berechnete Auflösungen": "pumped_storage_consumption_mwh",
    "Residuallast [MWh] Berechnete Auflösungen": "residual_load_mwh"
})

In [147]:
prices_clean = prices[
    [
        "Datum von",
        "Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen",
        "Belgien [€/MWh] Berechnete Auflösungen",
        "Dänemark 1 [€/MWh] Berechnete Auflösungen",
        "Frankreich [€/MWh] Berechnete Auflösungen",
        "Niederlande [€/MWh] Berechnete Auflösungen",
        "Österreich [€/MWh] Berechnete Auflösungen",
        "Polen [€/MWh] Berechnete Auflösungen",
        "Schweiz [€/MWh] Berechnete Auflösungen"
    ]
].copy()

In [149]:
prices_clean = prices_clean.rename(columns={
    "Datum von": "timestamp",
    "Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen": "price_germany",
    "Belgien [€/MWh] Berechnete Auflösungen": "price_belgium",
    "Dänemark 1 [€/MWh] Berechnete Auflösungen": "price_denmark",
    "Frankreich [€/MWh] Berechnete Auflösungen": "price_france",
    "Niederlande [€/MWh] Berechnete Auflösungen": "price_netherlands",
    "Österreich [€/MWh] Berechnete Auflösungen": "price_austria",
    "Polen [€/MWh] Berechnete Auflösungen": "price_poland",
    "Schweiz [€/MWh] Berechnete Auflösungen": "price_switzerland"
})
prices_clean.head()

,timestamp,price_germany,price_belgium,price_denmark,price_france,price_netherlands,price_austria,price_poland,price_switzerland
0,2021-01-01 00:00:00,50.87,50.87,50.87,50.87,50.87,50.87,34.18,50.98
1,2021-01-01 01:00:00,48.19,48.19,48.19,48.19,48.19,48.19,32.61,45.94
2,2021-01-01 02:00:00,44.68,44.68,44.68,44.68,44.68,44.68,32.58,44.53
3,2021-01-01 03:00:00,42.92,42.92,42.92,42.92,42.92,42.92,32.20,41.01
4,2021-01-01 04:00:00,40.39,40.39,40.39,40.39,40.39,40.39,29.16,39.23


In [151]:
generation_clean = generation_clean.rename(columns={
    "biomass_mwh": "biomass",
    "hydro_mwh": "hydro",
    "wind_offshore_mwh": "wind_offshore",
    "wind_onshore_mwh": "wind_onshore",
    "solar_mwh": "solar",
    "other_renewables_mwh": "other_renewables",
    "nuclear_mwh": "nuclear",
    "lignite_mwh": "lignite",
    "hard_coal_mwh": "hard_coal",
    "gas_mwh": "gas",
    "pumped_storage_generation_mwh": "pumped_storage_generation",
    "other_conventional_mwh": "other_conventional"
})

In [153]:
consumption_clean = consumption_clean.rename(columns={
    "load_mwh": "load",
    "load_incl_pumped_storage_mwh": "load_incl_pumped_storage",
    "pumped_storage_consumption_mwh": "pumped_storage_consumption",
    "residual_load_mwh": "residual_load"
})

In [155]:
print(generation_clean.columns.tolist())
print(consumption_clean.columns.tolist())

['timestamp', 'biomass', 'hydro', 'wind_offshore', 'wind_onshore', 'solar', 'other_renewables', 'nuclear', 'lignite', 'hard_coal', 'gas', 'pumped_storage_generation', 'other_conventional']
['timestamp', 'load', 'load_incl_pumped_storage', 'pumped_storage_consumption', 'residual_load']


In [157]:
#Handle the DST duplicate timestamps

for name, df in {
    "Generation": generation_clean,
    "Consumption": consumption_clean,
    "Prices": prices_clean
}.items():
    
    duplicate_counts = (
        df[df["timestamp"].duplicated(keep=False)]
        .groupby("timestamp")
        .size()
    )
    
    print(name)
    print(duplicate_counts)
    print()

Generation
timestamp
2021-10-31 02:00:00    2
2022-10-30 02:00:00    2
2023-10-29 02:00:00    2
2024-10-27 02:00:00    2
2025-10-26 02:00:00    2
dtype: int64

Consumption
timestamp
2021-10-31 02:00:00    2
2022-10-30 02:00:00    2
2023-10-29 02:00:00    2
2024-10-27 02:00:00    2
2025-10-26 02:00:00    2
dtype: int64

Prices
timestamp
2021-10-31 02:00:00    2
2022-10-30 02:00:00    2
2023-10-29 02:00:00    2
2024-10-27 02:00:00    2
2025-10-26 02:00:00    2
dtype: int64



In [161]:
for df in [generation_clean, consumption_clean, prices_clean]:
    df["dst_occurrence"] = df.groupby("timestamp").cumcount()

In [163]:
for name, df in {
    "Generation": generation_clean,
    "Consumption": consumption_clean,
    "Prices": prices_clean
}.items():
    
    duplicates = df.duplicated(
        subset=["timestamp", "dst_occurrence"]
    ).sum()
    
    print(name, ":", duplicates)

Generation : 0
Consumption : 0
Prices : 0


### 4.1 Daylight-Saving-Time Handling

The timestamp validation identified five repeated local timestamps in each dataset. These occur during the annual autumn daylight-saving-time transition, when the hour from 02:00 to 03:00 occurs twice.

Removing these observations would result in the loss of valid hourly electricity-market data. However, merging the datasets using the local timestamp alone could create multiple matches for the repeated hours.

To preserve both observations, an occurrence number is assigned to each timestamp using `cumcount()`. The combination of `timestamp` and `dst_occurrence` therefore provides a unique key for merging the generation, consumption, and price datasets.

A validation check confirmed that this combined key is unique in all three datasets.

The spring daylight-saving-time transition also produces a two-hour difference between consecutive local timestamps because the local clock skips from 01:00 to 03:00. These observations do not represent missing source data and are therefore not imputed.

Consequently, the dataset preserves the original local-time structure of the SMARD data, including 23-hour days during the spring transition and 25-hour days during the autumn transition.

In [166]:
electricity_data = generation_clean.merge(
    consumption_clean,
    on=["timestamp", "dst_occurrence"],
    how="inner"
)

In [168]:
print("Generation rows:", len(generation_clean))
print("Consumption rows:", len(consumption_clean))
print("Merged rows:", len(electricity_data))

Generation rows: 43824
Consumption rows: 43824
Merged rows: 43824


In [170]:
electricity_data = electricity_data.merge(
    prices_clean,
    on=["timestamp", "dst_occurrence"],
    how="inner"
)

In [172]:
print("Generation rows:", len(generation_clean))
print("Consumption rows:", len(consumption_clean))
print("Prices rows:", len(prices_clean))
print("Final merged rows:", len(electricity_data))

Generation rows: 43824
Consumption rows: 43824
Prices rows: 43824
Final merged rows: 43824


### 4.2 Dataset Integration

The cleaned generation, consumption, and wholesale price datasets were merged using the combination of `timestamp` and `dst_occurrence` as the merge key.

An inner join was used because all three datasets cover the same hourly observation period.

The number of observations remained unchanged throughout the integration:

- Generation: 43,824 observations
- Consumption: 43,824 observations
- Wholesale prices: 43,824 observations
- Final merged dataset: 43,824 observations

This confirms that all observations were successfully matched and that the repeated daylight-saving-time hours did not create duplicate or missing rows during the merge.

In [177]:
electricity_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 26 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   timestamp                   43824 non-null  datetime64[ns]
 1   biomass                     43824 non-null  float64       
 2   hydro                       43824 non-null  float64       
 3   wind_offshore               43824 non-null  float64       
 4   wind_onshore                43824 non-null  float64       
 5   solar                       43824 non-null  float64       
 6   other_renewables            43824 non-null  float64       
 7   nuclear                     43824 non-null  float64       
 8   lignite                     43824 non-null  float64       
 9   hard_coal                   43824 non-null  float64       
 10  gas                         43824 non-null  float64       
 11  pumped_storage_generation   43824 non-null  float64   

## 5. Export Cleaned Dataset

After completing data cleaning, validation, and integration, the merged dataset is exported for use in the subsequent feature-engineering and analysis stages.

This separates the data-cleaning workflow from the analytical workflow and avoids repeating the complete preprocessing procedure each time the project is opened.

In [181]:
electricity_data.to_csv(
    "../data/processed/electricity_data_clean.csv",
    index=False
)